# Smart Police Patrol v3 — XGBoost + CatBoost Ensemble
**24h forecast | H3 res-9 | Walk-forward | Learned blend weights**

Improvements over v2:
- **CatBoost** (Poisson) alongside XGBoost — diverse learners reduce variance
- **Hotspot-persistence features** — `was_top10_lag1/7`, `hotspot_streak_7d` (strongest P@10 signal)
- **Learned per-fold blend weight** — val-set MAE minimization picks XGB vs CAT weight each fold
- **Cell hotness rank** — `cell_pct_rank_7d`, `cell_pct_rank_28d` (relative standing vs peers)
- **Sliding + expanding hybrid** — last 12 weeks max to keep recent patterns dominant
- **Poisson-deviance early stopping** for both models
- **Rank-boosted final score** — blend prediction with rolling hotspot-rank for patrol allocation


## 0 · Setup

In [ ]:
import pandas as pd, numpy as np, json, h3, warnings, time
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar
warnings.filterwarnings('ignore')

CSV_PATH           = 'jan to may police violation_anonymized791b166.csv'
H3_RES             = 9
TZ                 = 'Asia/Kolkata'
MIN_CELL_EVENTS    = 50
MIN_TRAIN_WEEKS    = 4
TRAIN_WINDOW_WEEKS = 12   # sliding window: focus on recent 12 weeks
TOP_K              = 10   # hotspot threshold for persistence features
CALIB_FOLDS        = 6    # folds used to fit isotonic calibrator
RANDOM_STATE       = 42
t0 = time.time()


## 1 · Load & clean

In [ ]:
df = pd.read_csv(CSV_PATH, low_memory=False, usecols=[
    'id','latitude','longitude','created_datetime',
    'police_station','violation_type','vehicle_type'])
df['dt']   = pd.to_datetime(df['created_datetime'], errors='coerce', utc=True).dt.tz_convert(TZ)
df         = df.dropna(subset=['dt','latitude','longitude'])
df         = df[df.latitude.between(12.7, 13.2) & df.longitude.between(77.3, 77.9)]
df['date'] = df['dt'].dt.floor('D').dt.tz_localize(None)
df['cell'] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(df.latitude, df.longitude)]
print(f'rows={len(df):,}  cells={df.cell.nunique()}  '
      f'span={df.date.min().date()}..{df.date.max().date()}  ({round(time.time()-t0,1)}s)')


## 2 · Panel

In [ ]:
daily = df.groupby(['cell','date']).size().rename('count').reset_index()
vc    = df.cell.value_counts()
keep  = vc[vc >= MIN_CELL_EVENTS].index
print(f'modeled cells: {len(keep)} of {df.cell.nunique()}')

dates = pd.date_range(df.date.min(), df.date.max(), freq='D')
idx   = pd.MultiIndex.from_product([keep, dates], names=['cell','date'])
panel = daily.set_index(['cell','date']).reindex(idx, fill_value=0).reset_index()

stmap = df.groupby('cell')['police_station'].agg(lambda s: s.mode().iloc[0])
panel['station']  = panel['cell'].map(stmap).astype('category')
panel['station_s']= panel['cell'].map(stmap).astype(str)   # string version for CatBoost
panel = panel.sort_values(['cell','date']).reset_index(drop=True)
panel['week_idx'] = ((panel.date - panel.date.min()).dt.days // 7).astype(int)
print('panel shape', panel.shape, '| weeks', panel.week_idx.nunique())


## 3 · Target — 24h only

In [ ]:
g = panel.groupby('cell')['count']
panel['y_t1'] = g.shift(-1)


## 4 · Feature engineering

**4.1 Calendar + cyclical**

In [ ]:
import holidays
ka  = holidays.India(subdiv='KA', years=[2023, 2024])
hol = pd.to_datetime([d for d in ka
                       if panel.date.min() <= pd.Timestamp(d) <= panel.date.max()])

panel['dow']            = panel.date.dt.dayofweek
panel['is_weekend']     = (panel.dow >= 5).astype(int)
panel['month']          = panel.date.dt.month
panel['day_of_month']   = panel.date.dt.day
panel['quarter']        = panel.date.dt.quarter
panel['week_of_period'] = panel.week_idx
panel['is_month_start'] = (panel.day_of_month <= 3).astype(int)
panel['is_month_end']   = (panel.day_of_month >= 28).astype(int)
panel['dow_sin']        = np.sin(2*np.pi*panel.dow/7)
panel['dow_cos']        = np.cos(2*np.pi*panel.dow/7)
panel['month_sin']      = np.sin(2*np.pi*(panel.month-1)/12)
panel['month_cos']      = np.cos(2*np.pi*(panel.month-1)/12)
panel['is_holiday']     = panel.date.isin(hol).astype(int)
panel['days_from_holiday'] = panel.date.apply(
    lambda d: (d - hol[hol<=d]).days.min() if (hol<=d).any() else 999)
panel['days_to_holiday']   = panel.date.apply(
    lambda d: (hol[hol>=d] - d).days.min() if (hol>=d).any() else 999)
print('calendar done', round(time.time()-t0,1), 's')


**4.2 Lags + rolling + trend**

In [ ]:
g  = panel.groupby('cell')['count']
gp = panel.groupby('cell')['count']

for L in [1, 2, 3, 7, 14, 21, 28]:
    panel[f'lag_{L}'] = g.shift(L)

def roll(s, w, fn):
    return s.shift(1).rolling(w, min_periods=max(2, w//2)).agg(fn)

for w, fn in [(3,'mean'),(7,'mean'),(14,'mean'),(21,'mean'),
              (7,'std'),(14,'std'),(7,'max'),(7,'median')]:
    panel[f'roll_{fn}_{w}'] = gp.transform(lambda s, _w=w, _fn=fn: roll(s, _w, _fn))

panel['ewm_7']  = gp.transform(lambda s: s.shift(1).ewm(span=7,  min_periods=2).mean())
panel['ewm_14'] = gp.transform(lambda s: s.shift(1).ewm(span=14, min_periods=4).mean())

panel['accel']    = panel.roll_mean_3  - panel.roll_mean_7
panel['diff_1']   = panel.lag_1        - panel.lag_2
panel['diff_7']   = panel.lag_7        - panel.lag_14
prevwk  = gp.transform(lambda s: roll(s, 7, 'mean').shift(7))
prev4w  = gp.transform(lambda s: roll(s, 7, 'mean').shift(28))
panel['growth_7']  = (panel.roll_mean_7 - prevwk) / (prevwk + 1)
panel['growth_28'] = (panel.roll_mean_7 - prev4w) / (prev4w + 1)
print('lags done', round(time.time()-t0,1), 's')


**4.3 Same-weekday + cell expanding mean + peer rank**

In [ ]:
def past_dow_mean(d):
    d = d.sort_values('date')
    return d.groupby('dow')['count'].apply(lambda s: s.shift(1).expanding().mean())

panel['same_dow_mean']       = (panel.groupby('cell', group_keys=False)
                                     .apply(past_dow_mean)
                                     .reset_index(level=0, drop=True))
panel['cell_expanding_mean'] = gp.transform(lambda s: s.shift(1).expanding().mean())
panel['dev_from_dow']        = panel.lag_7 - panel.same_dow_mean

# how hot is this cell relative to all peers (pct rank of expanding mean)
panel['cell_rank']        = panel.groupby('date')['cell_expanding_mean'].rank(pct=True)

# rolling peer rank over 7d and 28d windows
wide_mean7 = panel.pivot_table(index='date', columns='cell', values='roll_mean_7', fill_value=0)
panel['cell_pct_rank_7d']  = (wide_mean7.rank(axis=1, pct=True)
                               .shift(1).stack().reindex(panel.set_index(['date','cell']).index)
                               .values)
wide_mean28 = panel.pivot_table(index='date', columns='cell', values='roll_mean_21', fill_value=0)
panel['cell_pct_rank_28d'] = (wide_mean28.rank(axis=1, pct=True)
                               .shift(1).stack().reindex(panel.set_index(['date','cell']).index)
                               .values)
print('cell rank done', round(time.time()-t0,1), 's')


**4.4 Hotspot persistence — strongest P@K signal**
Was this cell in the actual top-10 yesterday / last week? How many of last 7 days was it top-10?
All features are lagged so there is zero leakage.

In [ ]:
wide_count = panel.pivot_table(index='date', columns='cell', values='count', fill_value=0)

# actual daily top-K membership (1/0)
is_top = (wide_count.rank(axis=1, ascending=False, method='min') <= TOP_K).astype(int)

# lag-1 and lag-7 (leakage-safe)
top_lag1 = is_top.shift(1).fillna(0)
top_lag7 = is_top.shift(7).fillna(0)

# streak: how many of the 7 days before today was cell in top-K
streak_7 = is_top.shift(1).rolling(7, min_periods=1).sum()

# longest run in last 14d  
def longest_run(s):
    """rolling 14-day longest consecutive streak."""
    result = np.zeros(len(s))
    for i in range(len(s)):
        window = s[max(0,i-13):i]
        if len(window) == 0:
            result[i] = 0
            continue
        cur = best = 0
        for v in window:
            cur = cur+1 if v else 0
            best = max(best, cur)
        result[i] = best
    return result

def melt_join(wide_df, name):
    long = wide_df.stack().rename(name).reset_index()
    long.columns = ['date','cell',name]
    return long

panel = panel.merge(melt_join(top_lag1,  'was_top10_lag1'),     on=['date','cell'], how='left')
panel = panel.merge(melt_join(top_lag7,  'was_top10_lag7'),     on=['date','cell'], how='left')
panel = panel.merge(melt_join(streak_7,  'hotspot_streak_7d'), on=['date','cell'], how='left')

# fill any NaN from merge gaps
for c in ['was_top10_lag1','was_top10_lag7','hotspot_streak_7d']:
    panel[c] = panel[c].fillna(0)

print(f'hotspot features done  ({round(time.time()-t0,1)}s)')
print(f'  was_top10_lag1 prevalence: {panel.was_top10_lag1.mean():.3f}')
print(f'  streak mean (non-zero):    {panel.hotspot_streak_7d[panel.hotspot_streak_7d>0].mean():.2f}')


**4.5 Spatial — ring-1 + ring-2**

In [ ]:
nbrs1 = {c: [x for x in h3.grid_disk(c, 1) if x != c] for c in keep}
nbrs2 = {c: [x for x in h3.grid_disk(c, 2)
             if x != c and x not in set(nbrs1[c])] for c in keep}

wide = panel.pivot_table(index='date', columns='cell', values='count', fill_value=0)
prev = wide.shift(1)

def ring_sum(nbr_dict, name):
    ns = {}
    for c, nn in nbr_dict.items():
        cols = [x for x in nn if x in prev.columns]
        ns[c] = prev[cols].sum(axis=1) if cols else pd.Series(0, index=prev.index)
    df_ = pd.DataFrame(ns).stack().rename(name).reset_index()
    df_.columns = ['date','cell',name]
    return df_

panel = panel.merge(ring_sum(nbrs1, 'nbr_count_lag1'),  on=['date','cell'], how='left')
panel = panel.merge(ring_sum(nbrs2, 'nbr_ring2_lag1'), on=['date','cell'], how='left')
print(f'spatial done  ({round(time.time()-t0,1)}s)')


**4.6 Composition shares**

In [ ]:
BUCKET = {'WRONG PARKING':'wrong','NO PARKING':'no_parking',
          'PARKING IN A MAIN ROAD':'road','PARKING NEAR ROAD CROSSING':'road',
          'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS':'road','PARKING ON FOOTPATH':'footpath'}
def primary(x):
    try: t = json.loads(x)
    except: return 'other'
    return BUCKET.get(t[0], 'other') if t else 'other'

df['vbucket'] = df['violation_type'].apply(primary)
df['is2w']    = df['vehicle_type'].isin(['SCOOTER','MOTOR CYCLE','MOPED'])

shares = df.groupby(['cell','date']).agg(
    share_wrong   =('vbucket', lambda s:(s=='wrong').mean()),
    share_nopark  =('vbucket', lambda s:(s=='no_parking').mean()),
    share_2wheeler=('is2w','mean')).reset_index()
panel = panel.merge(shares, on=['cell','date'], how='left')
for col in ['share_wrong','share_nopark','share_2wheeler']:
    panel[col] = (panel.groupby('cell')[col]
                       .transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean()))

print(f'all features done — cols={panel.shape[1]}  ({round(time.time()-t0,1)}s)')


## 5 · Walk-forward — XGBoost + CatBoost ensemble
Per fold:
1. Split training into `tr2` (train) + `val` (last week, for early stopping + blend weight)
2. Fit XGBoost and CatBoost independently with early stopping
3. Learn per-fold blend weight `w` that minimises val-set MAE
4. Apply `w * xgb + (1-w) * cat` on test week

In [ ]:
FEATS_NUM = [
    # calendar
    'dow','is_weekend','month','day_of_month','quarter','week_of_period',
    'is_month_start','is_month_end',
    'dow_sin','dow_cos','month_sin','month_cos',
    'is_holiday','days_to_holiday','days_from_holiday',
    # lags
    'lag_1','lag_2','lag_3','lag_7','lag_14','lag_21','lag_28',
    # rolling
    'roll_mean_3','roll_mean_7','roll_mean_14','roll_mean_21',
    'roll_std_7','roll_std_14','roll_max_7','roll_median_7',
    'ewm_7','ewm_14',
    # trend
    'accel','diff_1','diff_7','growth_7','growth_28',
    # cell-level
    'same_dow_mean','cell_expanding_mean','dev_from_dow',
    'cell_rank','cell_pct_rank_7d','cell_pct_rank_28d',
    # hotspot persistence
    'was_top10_lag1','was_top10_lag7','hotspot_streak_7d',
    # spatial
    'nbr_count_lag1','nbr_ring2_lag1',
    # composition
    'share_wrong','share_nopark','share_2wheeler',
]
FEATS_XGB = FEATS_NUM + ['station']       # XGBoost: pandas category dtype
FEATS_CAT = FEATS_NUM + ['station_s']     # CatBoost: string column
CAT_IDX   = [FEATS_CAT.index('station_s')]

def xgb_model():
    return xgb.XGBRegressor(
        objective='count:poisson', n_estimators=400, learning_rate=0.05,
        max_depth=5, min_child_weight=3, gamma=0.05,
        subsample=0.8, colsample_bytree=0.7, colsample_bylevel=0.8,
        reg_alpha=0.1, reg_lambda=1.5,
        tree_method='hist', enable_categorical=True,
        early_stopping_rounds=25, random_state=RANDOM_STATE, verbosity=0)

def cat_model():
    return CatBoostRegressor(
        loss_function='Poisson', iterations=400, learning_rate=0.05,
        depth=5, min_data_in_leaf=3, l2_leaf_reg=3.0,
        subsample=0.8, colsample_bylevel=0.7,
        cat_features=CAT_IDX, random_seed=RANDOM_STATE,
        early_stopping_rounds=25, verbose=0)

data = panel.dropna(subset=['y_t1']).copy()
data['station']   = data['station'].astype('category')
data['station_s'] = data['station_s'].astype(str)

weeks = sorted(data.week_idx.unique())
preds, blend_weights, best_iters_xgb, best_iters_cat = [], [], [], []

for w in weeks:
    if w < MIN_TRAIN_WEEKS:
        continue

    # sliding window
    w_start = max(0, w - TRAIN_WINDOW_WEEKS) if TRAIN_WINDOW_WEEKS else 0
    tr = data[(data.week_idx >= w_start) & (data.week_idx < w)].copy()
    te = data[data.week_idx == w].copy()
    if len(te) == 0: continue

    # align XGB categories
    all_cats = tr['station'].cat.categories.union(te['station'].cat.categories)
    tr['station'] = tr['station'].cat.set_categories(all_cats)
    te['station'] = te['station'].cat.set_categories(all_cats)

    # hold out last week for early stopping + blend weight learning
    val_w = tr.week_idx.max()
    val   = tr[tr.week_idx == val_w].copy()
    tr2   = tr[tr.week_idx <  val_w].copy()
    val['station'] = val['station'].cat.set_categories(all_cats)
    tr2['station'] = tr2['station'].cat.set_categories(all_cats)

    # ── XGBoost ──────────────────────────────────────────────────────────────
    mx = xgb_model()
    if len(tr2) > 0:
        mx.fit(tr2[FEATS_XGB], tr2['y_t1'],
               eval_set=[(val[FEATS_XGB], val['y_t1'])], verbose=False)
    else:
        mx.fit(tr[FEATS_XGB], tr['y_t1'])
    best_iters_xgb.append(getattr(mx, 'best_iteration', None))

    # ── CatBoost ─────────────────────────────────────────────────────────────
    mc = cat_model()
    if len(tr2) > 0:
        mc.fit(tr2[FEATS_CAT], tr2['y_t1'],
               eval_set=(val[FEATS_CAT], val['y_t1']),
               use_best_model=True, verbose=False)
    else:
        mc.fit(tr[FEATS_CAT], tr['y_t1'])
    best_iters_cat.append(mc.best_iteration_ if hasattr(mc, 'best_iteration_') else None)

    # ── Learn blend weight from val set ──────────────────────────────────────
    xgb_val = np.clip(mx.predict(val[FEATS_XGB]), 0, None)
    cat_val  = np.clip(mc.predict(val[FEATS_CAT]), 0, None)
    y_val    = val['y_t1'].values
    if len(y_val) > 10:
        res = minimize_scalar(
            lambda alpha: np.abs(alpha*xgb_val + (1-alpha)*cat_val - y_val).mean(),
            bounds=(0, 1), method='bounded')
        w_blend = float(np.clip(res.x, 0.2, 0.8))  # prevent degenerate weighting
    else:
        w_blend = 0.5
    blend_weights.append(w_blend)

    # ── Predict test week ────────────────────────────────────────────────────
    xgb_pred = np.clip(mx.predict(te[FEATS_XGB]), 0, None)
    cat_pred  = np.clip(mc.predict(te[FEATS_CAT]), 0, None)
    ens_pred  = w_blend * xgb_pred + (1 - w_blend) * cat_pred

    out = te[['cell','date','week_idx','y_t1','lag_7','cell_expanding_mean']].copy()
    out['pred']     = ens_pred
    out['pred_xgb'] = xgb_pred
    out['pred_cat'] = cat_pred
    out['w_xgb']    = w_blend
    out['naive']    = out['lag_7'].fillna(out['cell_expanding_mean']).fillna(0)
    preds.append(out)

preds = pd.concat(preds, ignore_index=True)
xi = [x for x in best_iters_xgb if x is not None]
ci = [x for x in best_iters_cat if x is not None]
print(f'walk-forward done: {preds.week_idx.nunique()} batches, {len(preds):,} predictions')
print(f'XGB best_iter  avg={np.mean(xi):.0f}  min={min(xi)}  max={max(xi)}')
print(f'CAT best_iter  avg={np.mean(ci):.0f}  min={min(ci)}  max={max(ci)}')
print(f'blend w_xgb    avg={np.mean(blend_weights):.3f}  '
      f'min={min(blend_weights):.2f}  max={max(blend_weights):.2f}')


## 6 · Isotonic calibration on early folds

In [ ]:
fold_weeks  = sorted(preds.week_idx.unique())
calib_weeks = fold_weeks[:CALIB_FOLDS]
eval_weeks  = fold_weeks[CALIB_FOLDS:]

calib_df = preds[preds.week_idx.isin(calib_weeks)]
eval_df  = preds[preds.week_idx.isin(eval_weeks)].copy()

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(calib_df['pred'].values, calib_df['y_t1'].values)
eval_df['pred_raw'] = eval_df['pred']
eval_df['pred']     = np.clip(iso.predict(eval_df['pred'].values), 0, None)

print(f'calibration: {len(calib_weeks)} folds fit, {len(eval_weeks)} folds evaluated')
print(f'  raw MAE = {np.abs(eval_df.pred_raw - eval_df.y_t1).mean():.3f}')
print(f'  cal MAE = {np.abs(eval_df.pred     - eval_df.y_t1).mean():.3f}')


## 7 · Evaluation

In [ ]:
def poisson_dev(y, p):
    p = np.clip(p, 1e-6, None)
    return 2*np.mean(np.where(y>0, y*np.log(y/p), 0) - (y-p))

def precision_at_k(frame, k=10, pred_col='pred'):
    hits = []
    for _, g in frame.groupby('date'):
        if len(g) < k: continue
        hits.append(len(set(g.nlargest(k,'y_t1').cell)
                      & set(g.nlargest(k, pred_col).cell)) / k)
    return np.mean(hits)

y, p, px, pc, nv = (preds.y_t1.values, preds.pred.values,
                    preds.pred_xgb.values, preds.pred_cat.values,
                    preds.naive.values)

print('=== Full walk-forward ===')
print(f'         ensemble    xgb-only   cat-only    naive')
print(f'MAE    {np.abs(p-y).mean():8.3f}  {np.abs(px-y).mean():8.3f}  '
      f'{np.abs(pc-y).mean():8.3f}  {np.abs(nv-y).mean():8.3f}')
print(f'RMSE   {np.sqrt(((p-y)**2).mean()):8.3f}  {np.sqrt(((px-y)**2).mean()):8.3f}  '
      f'{np.sqrt(((pc-y)**2).mean()):8.3f}  {np.sqrt(((nv-y)**2).mean()):8.3f}')
print(f'PDev   {poisson_dev(y,p):8.3f}  {poisson_dev(y,px):8.3f}  '
      f'{poisson_dev(y,pc):8.3f}  {poisson_dev(y,nv):8.3f}')
print(f'P@10   {precision_at_k(preds):8.3f}  '
      f'{precision_at_k(preds,pred_col="pred_xgb"):8.3f}  '
      f'{precision_at_k(preds,pred_col="pred_cat"):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive)):8.3f}')
print(f'P@25   {precision_at_k(preds,25):8.3f}  '
      f'{precision_at_k(preds,25,"pred_xgb"):8.3f}  '
      f'{precision_at_k(preds,25,"pred_cat"):8.3f}  '
      f'{precision_at_k(preds.assign(pred=preds.naive),25):8.3f}')

if len(eval_weeks) > 0:
    ey, ep = eval_df.y_t1.values, eval_df.pred.values
    print(f'\n=== Calibrated eval folds ({len(eval_weeks)} weeks) ===')
    print(f'MAE {np.abs(ep-ey).mean():.3f}  |  '
          f'P@10 {precision_at_k(eval_df):.3f}  |  '
          f'P@25 {precision_at_k(eval_df,25):.3f}')

wk = (preds.assign(ae=np.abs(preds.pred-preds.y_t1),
                   ae_n=np.abs(preds.naive-preds.y_t1))
           .groupby('week_idx').agg(n=('y_t1','size'),
                                    mae_ens=('ae','mean'),
                                    mae_naive=('ae_n','mean')).round(3))
wk


## 8 · Severity + rank-boosted patrol allocation (24h)

In [ ]:
latest = preds[preds.week_idx == preds.week_idx.max()].copy()

q = preds.y_t1.quantile([0.50, 0.80, 0.95]).values
def severity(c):
    if c <= q[0]: return 'Low'
    if c <= q[1]: return 'Medium'
    if c <= q[2]: return 'High'
    return 'Critical'
OFFICERS = {'Low':1, 'Medium':2, 'High':3, 'Critical':4}

latest['severity'] = latest['pred'].apply(severity)
latest['officers'] = latest['severity'].map(OFFICERS)

latest = latest.merge(
    panel[['cell','date','growth_7','nbr_count_lag1','nbr_ring2_lag1',
           'hotspot_streak_7d','cell_pct_rank_7d','was_top10_lag1']],
    on=['cell','date'], how='left')

# rank-boosted risk: count x trend x spatial pressure x persistence
nbr1_mean = latest['nbr_count_lag1'].mean() + 1
nbr2_mean = latest['nbr_ring2_lag1'].mean() + 1
latest['risk_score'] = (
    latest['pred']
    * (1 + latest['growth_7'].fillna(0).clip(-0.9, None))
    * (1 + latest['nbr_count_lag1'].fillna(0) / nbr1_mean)
    * (1 + 0.3 * latest['nbr_ring2_lag1'].fillna(0) / nbr2_mean)
    * (1 + 0.5 * latest['was_top10_lag1'].fillna(0))          # +50% if was hotspot yesterday
    * (1 + 0.1 * latest['hotspot_streak_7d'].fillna(0))       # +10% per streak day
)

print('severity cutoffs (p50/p80/p95):', q.round(2))
print(f'\nTop 10 hotspots — {latest.date.max().date()} (24h forecast)')
latest.sort_values('risk_score', ascending=False).head(10)[
    ['cell','pred','severity','officers','risk_score','was_top10_lag1','hotspot_streak_7d']].round(2)


## 9 · Save model + outputs

In [ ]:
import joblib, os
os.makedirs('model_artifacts', exist_ok=True)

mx.save_model('model_artifacts/xgb_v3.json')
mc.save_model('model_artifacts/cat_v3.cbm')
joblib.dump({
    'feats_xgb': FEATS_XGB, 'feats_cat': FEATS_CAT,
    'cat_idx': CAT_IDX, 'station_cats': all_cats,
    'h3_res': H3_RES, 'severity_quantiles': q.tolist(),
    'isotonic': iso,
    'blend_weights': blend_weights,
    'avg_w_xgb': float(np.mean(blend_weights))
}, 'model_artifacts/model_v3_meta.pkl')

out_cols = ['cell','date','pred','pred_xgb','pred_cat','severity','officers','risk_score']
latest.sort_values('risk_score', ascending=False)[out_cols].round(3).to_csv(
    'patrol_forecast_v3_latest.csv', index=False)
try:
    preds.to_parquet('walkforward_v3_predictions.parquet', index=False)
    print('saved parquet')
except:
    preds.to_csv('walkforward_v3_predictions.csv', index=False)
print('model_artifacts/xgb_v3.json + cat_v3.cbm + model_v3_meta.pkl')
print('patrol_forecast_v3_latest.csv')


## 10 · Unseen test map (last 4 weeks)

In [ ]:
TEST_WEEKS = 4

def build_test_map(preds, test_weeks=TEST_WEEKS, out_html='patrol_map_v3_test.html',
                   center=(12.9716, 77.5946), zoom=11):
    wmax = preds.week_idx.max()
    test = preds[preds.week_idx >= wmax - test_weeks + 1].copy()
    d0, d1 = pd.to_datetime(test.date).min(), pd.to_datetime(test.date).max()
    agg = test.groupby('cell').agg(pred=('pred','sum'), actual=('y_t1','sum')).reset_index()
    agg['err'] = agg.pred - agg.actual
    mae = np.abs(test.pred - test.y_t1).mean()
    def p_at_k(frame, k=10):
        hits=[]
        for _, g in frame.groupby('date'):
            if len(g) < k: continue
            hits.append(len(set(g.nlargest(k,'y_t1').cell) & set(g.nlargest(k,'pred').cell))/k)
        return np.mean(hits)
    p10 = p_at_k(test, 10)
    p25 = p_at_k(test, 25)
    qs   = np.quantile(agg.actual, [0,.2,.4,.6,.8,1.0])
    RAMP = ['#fee5d9','#fcae91','#fb6a4a','#de2d26','#a50f15']
    def color(v):
        for i in range(5):
            if v <= qs[i+1]: return RAMP[i]
        return RAMP[-1]
    def layer(valcol):
        feats=[]
        for _, r in agg.iterrows():
            b    = h3.cell_to_boundary(r['cell'])
            ring = [[lng,lat] for lat,lng in b] + [[b[0][1], b[0][0]]]
            feats.append({"type":"Feature","properties":{
                "cell":r['cell'],"pred":round(float(r.pred),1),
                "actual":round(float(r.actual),1),"err":round(float(r.err),1),
                "color":color(r[valcol])},
                "geometry":{"type":"Polygon","coordinates":[ring]}})
        return {"type":"FeatureCollection","features":feats}
    gj_pred, gj_act = layer('pred'), layer('actual')
    legend = "".join(
        f'<div><span style="background:{c}"></span>{int(qs[i])}-{int(qs[i+1])}</div>'
        for i, c in enumerate(RAMP))
    html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>v3 Unseen Test</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
html,body,#map{{height:100%;margin:0}}
.panel{{position:absolute;top:10px;left:50px;z-index:1000;background:#fff;padding:8px 12px;
  border-radius:8px;font:13px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3);max-width:400px}}
.legend{{position:absolute;bottom:20px;right:12px;z-index:1000;background:#fff;padding:10px 12px;
  border-radius:8px;font:12px system-ui;box-shadow:0 1px 6px rgba(0,0,0,.3)}}
.legend div{{display:flex;align-items:center;gap:6px;margin:2px 0}}
.legend span{{width:14px;height:14px;border-radius:3px;display:inline-block}}
</style></head><body>
<div id="map"></div>
<div class="panel"><b>v3 — XGB + CatBoost Ensemble (24h)</b><br>
{d0.date()} to {d1.date()} ({test_weeks} weeks, unseen)<br>
MAE {mae:.2f} | P@10 {p10:.0%} | P@25 {p25:.0%}<br>
<small>Toggle Predicted/Actual (top-right). Matching colours = good forecast.</small></div>
<div class="legend"><b>Violations / window</b>{legend}</div>
<script>
const map=L.map('map').setView([{center[0]},{center[1]}],{zoom});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png',
  {{maxZoom:19,attribution:'(c) OpenStreetMap'}}).addTo(map);
function mk(data){{return L.geoJSON(data,{{
  style:f=>({{color:'#333',weight:1,fillColor:f.properties.color,fillOpacity:0.6}}),
  onEachFeature:(f,l)=>{{const p=f.properties;
    l.bindPopup(`<b>${{p.cell}}</b><br>Predicted: ${{p.pred}}<br>Actual: ${{p.actual}}<br>Error: ${{p.err}}`);}}}})
}}}}
const predL=mk({json.dumps(gj_pred)}),actL=mk({json.dumps(gj_act)});
actL.addTo(map);
L.control.layers(null,{{"Actual (truth)":actL,"Predicted (model)":predL}},{{collapsed:false}}).addTo(map);
</script></body></html>"""
    with open(out_html,'w', encoding='utf-8') as f: f.write(html)
    print(f'map: {d0.date()}..{d1.date()} | {len(agg)} cells | '
          f'MAE {mae:.2f} | P@10 {p10:.0%} | P@25 {p25:.0%} -> {out_html}')

build_test_map(preds)
from IPython.display import IFrame
IFrame('patrol_map_v3_test.html', width='100%', height=560)


## 11 · Feature importance — XGBoost

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fi_xgb = pd.Series(mx.feature_importances_, index=FEATS_XGB).sort_values(ascending=False)
fi_cat = pd.Series(mc.get_feature_importance(), index=FEATS_CAT).sort_values(ascending=False)

print('Top 15 XGBoost features:')
print(fi_xgb.head(15).round(4).to_string())
print('\nTop 15 CatBoost features:')
print(fi_cat.head(15).round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fi_xgb.head(20).sort_values().plot.barh(ax=axes[0], title='XGBoost v3')
fi_cat.head(20).sort_values().plot.barh(ax=axes[1], title='CatBoost v3')
for ax in axes: ax.set_xlabel('importance')
plt.tight_layout()
plt.savefig('feature_importance_v3.png', dpi=120)
print('saved feature_importance_v3.png')
